In [ ]:
"""
教學範例：交叉式 Gage R&R + SPC 製程能力分析

資料為程式產生的模擬數據，不能當成實際量產結論。
設計：10 個零件 × 3 位作業員 × 2 次重複量測
製程：25 個子組 × 每組 4 件
"""
import numpy as np  # 匯入 numpy 套件並簡稱為 np，用於數值運算與隨機數產生
import pandas as pd  # 匯入 pandas 套件並簡稱為 pd，用於建立與整理表格資料 DataFrame
import matplotlib.pyplot as plt  # 匯入 matplotlib 繪圖模組，用於後續畫 Gage R&R 與 SPC 管制圖
import statsmodels.formula.api as smf  # 匯入 statsmodels 公式 API，用於建立變異數分析的線性模型
from statsmodels.stats.anova import anova_lm  # 從 statsmodels 匯入 anova_lm 函數，用於計算 ANOVA 表

# ----------------------------- 
# 設定：可依實際案例修改  # 說明註解：以下參數可依真實案例自行調整
# -----------------------------  
SEED = 2026  # 設定隨機種子為 2026，確保每次執行產生的模擬數據都一樣，可重現結果
LSL = 19.900  # 設定規格下限 Lower Spec Limit，產品尺寸不可小於 19.9
USL = 20.100  # 設定規格上限 Upper Spec Limit，產品尺寸不可大於 20.1
TARGET = 20.000  # 設定目標值 Target，即設計中心值 20.0

N_PARTS = 10  # 設定 Gage R&R 零件數為 10，對應設計中的 10 個零件
N_OPERATORS = 3  # 設定作業員人數為 3，對應設計中的 3 位作業員
N_REPEATS = 2  # 設定每人每件重複量測 2 次，用於評估重複性 Repeatability

N_SUBGROUPS = 25  # 設定 SPC 子組數為 25組，一般管制圖建議 25 組以上才穩定
SUBGROUP_SIZE = 4  # 設定每子組抽 4 件，即 Xbar-R 管制圖常見的 n=4

rng = np.random.default_rng(SEED)  # 用上面設定的 SEED 建立 numpy 新版隨機數產生器，後續模擬數據都用它來抽樣

In [ ]:
# ========================================================= 
# A. 產生交叉式 MSA（Gage R&R）模擬資料  # 標題註解：說明本段要產生 10零件×3人員×2重複=60筆量測數據
# ========================================================= 
parts = [f"P{i:02d}" for i in range(1, N_PARTS + 1)]  # 用列表生成式產生零件編號 P01~P10，02d表示補零兩位數
operators = ["A", "B", "C"]  # 定義3位作業員代號，對應前面設定的 N_OPERATORS=3

# 零件真值刻意分布在製程範圍內，以呈現零件間差異  # 說明註解：真值要分散，才能算出 Part-to-Part 變異
part_true = dict(zip(parts, np.linspace(19.930, 20.070, N_PARTS)))  # 用 linspace 在19.93~20.07間產生10個等距真值，再用 zip+dict 綁定 P01~P10

# 模擬作業員間的小幅平均差異  # 說明註解：模擬再現性 Reproducibility，即不同人量有系統性偏差
operator_effect = {"A": -0.0010, "B": 0.0000, "C": 0.0010}  # 用字典設定3人偏差值，A偏小、B置中、C偏大，單位與量測值相同

# 模擬 Part × Operator 交互作用  # 說明註解：模擬某人量某件特別偏大或偏小，ANOVA 才能拆出交互作用項
interaction_effect = {  # 開始建立字典：key是(零件,作業員)元組，value是該組合的偏差量，共10*3=30組
    (part, operator): rng.normal(0, 0.0007)  # 對每種組合從常態分配抽樣，平均0、標準差0.0007，作為交互作用誤差
    for part in parts  # 外層迴圈：遍歷 P01~P10 所有零件
    for operator in operators  # 內層迴圈：遍歷 A,B,C 所有作業員，形成交叉 Crossed 設計
}  # 字典生成式結束符號

msa_rows = []  # 建立空列表，後續每筆量測結果以字典形式暫存於此
for part in parts:  # 外層迴圈：依序處理每一個零件 P01~P10
    for operator in operators:  # 中層迴圈：每個零件都給 A,B,C 三人各量一次，確保交叉式設計
        for repeat in range(1, N_REPEATS + 1):  # 內層迴圈：每人每件重複量 N_REPEATS=2 次，repeat 為 1,2
            measured = (  # 開始計算本次模擬量測值，左括號允許換行撰寫
                part_true[part]  # 第1項：加上該零件的真值，為量測的基準
                + operator_effect[operator]  # 第2項：加上該作業員的系統偏差，模擬再現性
                + interaction_effect[(part, operator)]  # 第3項：加上該人×該件的交互作用偏差
                + rng.normal(0, 0.0012)  # 重複性誤差  # 第4項：每次重複都重新抽樣的隨機誤差，標準差0.0012，模擬重複性
            )  # 右括號：量測值四項加總結束
            msa_rows.append(  # 將本筆結果加入列表，開始呼叫 append
                {  # 開始建立一筆資料字典
                    "part": part,  # 欄位1：記錄零件編號
                    "operator": operator,  # 欄位2：記錄作業員代號
                    "repeat": repeat,  # 欄位3：記錄第幾次重複
                    "measurement": measured,  # 欄位4：記錄上面算出的模擬量測值
                }  # 字典結束
            )  # append 呼叫結束

msa = pd.DataFrame(msa_rows)  # 將 60 筆字典列表轉為 pandas DataFrame 表格，方便後續 ANOVA 分析

# 模擬量測順序隨機化  # 說明註解：實務 MSA 要求隨機順序量測，避免記憶偏差
msa = msa.sample(frac=1, random_state=SEED).reset_index(drop=True)  # 將表格打亂 frac=1表全部抽樣，用 SEED 固定亂序結果，再重設索引 0~59
msa["run_order"] = np.arange(1, len(msa) + 1)  # 新增 run_order 欄位，依打亂後順序編號 1~60，記錄實際量測順序
msa.to_csv("msa_synthetic_data.csv", index=False)  # 將 MSA 表格存檔為 CSV，index=False 表示不存多餘的索引欄

In [ ]:
msa

In [ ]:
# =========================================================  
# B. 交叉式 Gage R&R：ANOVA 及變異成分  # 標題註解：本段用 ANOVA 把總變異拆成零件、人員、交互作用、重複性
# =========================================================  
# 平衡交叉設計：part、operator 及其交互作用  # 說明註解：平衡指每人每件次數相同，交叉指每人量過所有零件
model = smf.ols(  # 開始用 statsmodels 公式建立普通最小平方 OLS 線性模型
    "measurement ~ C(part) * C(operator)",  # 模型公式：量測值受零件主效應、人員主效應及兩者交互作用影響，C()表示視為類別因子
    data=msa  # 指定分析資料來源為前面產生的 msa 表格
).fit()  # 呼叫 fit() 執行迴歸配適，結果存入 model 物件

anova_table = anova_lm(model, typ=2)  # 用 anova_lm 對配適模型做 Type II 變異數分析，產生含 sum_sq平方和、df自由度的表格

# statsmodels 沒有直接輸出 mean_sq，需自行計算：均方 = 平方和 / 自由度  # 說明註解：Mean Square = SS/df，後續估變異成分必須用 MS
anova_table["mean_sq"] = anova_table["sum_sq"] / anova_table["df"]  # 新增 mean_sq 欄：平方和除以自由度，得到各來源的均方

ms_part = anova_table.loc["C(part)", "mean_sq"]  # 取出零件因子的均方 MS_Part，反映零件間差異+誤差
ms_operator = anova_table.loc["C(operator)", "mean_sq"]  # 取出作業員因子的均方 MS_Operator，反映人員間差異+誤差
ms_interaction = anova_table.loc["C(part):C(operator)", "mean_sq"]  # 取出零件×人員交互作用的均方 MS_Part*Op
ms_repeatability = anova_table.loc["Residual", "mean_sq"]  # 取出殘差均方 MS_Error，即同一人同件重複量測的變異，視為重複性

# 平衡交叉 ANOVA 的方法矩估計；負值截為 0  # 說明註解：用期望均方 EMS 反解變異數，理論上變異數不為負，負值以0計
var_repeatability = max(ms_repeatability, 0.0)  # 重複性變異 = MS_Error，若為負則取0，max()做截斷
var_part_operator = max(  # 開始計算人員×零件交互作用變異，公式需跨行故用括號換行
    (ms_interaction - ms_repeatability) / N_REPEATS,  # 公式：(MS_交互-MS_誤差)/重複次數，扣除重複性後平均到每次重複
    0.0  # 下限0：若相減為負表示估計為0，取0避免負變異數
)  # 變異成分 max() 計算結束
var_operator = max(  # 開始計算作業員主效應變異，即單純人員差異
    (ms_operator - ms_interaction) / (N_PARTS * N_REPEATS),  # 公式：(MS_人員-MS_交互)/(零件數×重複數)，扣除交互作用後平均
    0.0  # 下限0：負值截為0
)  # max()計算結束
var_part = max(  # 開始計算零件主效應變異，即真正的產品差異 Part-to-Part
    (ms_part - ms_interaction) / (N_OPERATORS * N_REPEATS),  # 公式：(MS_零件-MS_交互)/(人數×重複數)，扣除交互作用後平均
    0.0  # 下限0：負值截為0
)  # max()計算結束

var_reproducibility = var_operator + var_part_operator  # 再現性變異 = 人員變異 + 交互作用變異，即換人量的總差異
var_grr = var_repeatability + var_reproducibility  # 量測系統總變異 GRR = 重複性 + 再現性
var_total = var_grr + var_part  # 全變異 = 量測變異 GRR + 零件變異，作為分母算 %StudyVar

sd_repeatability = np.sqrt(var_repeatability)  # 開根號將重複性變異數轉為標準差，單位回到量測單位，方便×6算寬度
sd_operator = np.sqrt(var_operator)  # 開根號將人員變異數轉為人員標準差
sd_interaction = np.sqrt(var_part_operator)  # 開根號將交互作用變異數轉為交互作用標準差，注意變數名簡寫為 sd_interaction
sd_grr = np.sqrt(var_grr)  # 開根號將 GRR 變異數轉為 GRR 標準差，為後續 %GRR 核心分子
sd_part = np.sqrt(var_part)  # 開根號將零件變異數轉為零件標準差，反映真實產品分散程度
sd_total = np.sqrt(var_total)  # 開根號將全變異數轉為總標準差，作為 %StudyVar 分母

tolerance = USL - LSL  # 計算規格公差寬度 = 上限-下限，本例 20.1-19.9=0.2，作為 %Tolerance 分母

pct_study_grr = (  # 開始計算 %StudyVar：GRR 佔全變異的百分比，評估量測佔整體波動比例
    100 * sd_grr / sd_total if sd_total > 0 else np.nan  # 條件式：若總標準差>0則算 100*SD_GRR/SD_Total，否則回傳 NaN 避免除以零
)  # %StudyVar 括號結束
pct_tolerance_grr = (  # 開始計算 %Tolerance：GRR 佔規格公差的百分比，評估量測吃掉多少規格
    100 * 6 * sd_grr / tolerance if tolerance > 0 else np.nan  # 公式：100*6*SD_GRR/公差，6倍標準差表99.73%範圍，若公差<=0則回傳 NaN
)  # %Tolerance 括號結束
ndc = (  # 開始計算 NDC 可區分類別數 Number of Distinct Categories，評估量具解析力
    int(np.floor(1.41 * sd_part / sd_grr))  # 公式：無條件捨去(1.41*SD_Part/SD_GRR)再轉整數，1.41約為 sqrt(2)，SD_GRR需>0
    if sd_grr > 0 else np.inf  # 條件式：若 SD_GRR>0才計算，否則回傳無限大 inf，避免除以零
)  # NDC 括號結束

if pct_tolerance_grr < 10 and ndc >= 5:  # 判斷式1：若%公差<10%且NDC>=5，表量測精良
    msa_judgment = "一般可接受（此處為課堂判讀規則）"  # 滿足嚴格標準時的結論字串，括號提醒此為簡化教學規則
elif pct_tolerance_grr <= 30 and ndc >= 5:  # 判斷式2：若%公差10~30%且NDC>=5，表邊緣可用
    msa_judgment = "有條件使用；應評估用途與風險"  # 次級結論：需看量測用途決定是否放行
else:  # 以上皆不滿足，即%公差>30%或NDC<5
    msa_judgment = "不建議直接發布製程能力結論；先檢討量測系統"  # 最差結論：量測太差，先改善量具再談 Cp/Cpk

In [ ]:
pct_study_grr

In [ ]:
# =========================================================  # 分隔線：標示 C 段落開始，美觀用
# C. 產生製程能力分析用的子組資料  # 標題：本段模擬 25組×4件製程數據，再算 Xbar-R 管制界限與 Cp/Cpk/Pp/Ppk
# =========================================================  # 分隔線：標示標題結束，美觀用
# 模擬穩定、近似常態的製程資料  # 說明：強調為受控、常態的理想數據，才能直接示範能力指標算法
process_values = rng.normal(  # 用隨機產生器抽常態亂數，結果為 2D 陣列
    loc=20.010,  # 平均數設 20.01，刻意偏離目標 20.0，用來示範 Cp 與 Cpk 差異
    scale=0.018,  # 標準差設 0.018，控制製程分散程度
    size=(N_SUBGROUPS, SUBGROUP_SIZE)  # 形狀 (25,4)：25個子組，每組4件
)  # rng.normal() 呼叫結束

process_rows = []  # 建空列表，暫存轉換後的每筆長格式資料
for subgroup_index, row in enumerate(process_values, start=1):  # 外圈：遍歷25列，編號1~25，row為該組4筆值
    for within_index, value in enumerate(row, start=1):  # 內圈：遍歷組內4值，編號1~4，value為量測值
        process_rows.append(  # 將本筆加入列表
            {  # 建一筆字典
                "subgroup": subgroup_index,  # 欄1：子組編號 1~25
                "within_subgroup": within_index,  # 欄2：組內順序 1~4
                "measurement": value,  # 欄3：該件量測值
            }  # 字典結束
        )  # append 結束

process = pd.DataFrame(process_rows)  # 將100筆字典轉為 DataFrame 表格
process.to_csv("process_synthetic_data.csv", index=False)  # 存檔為 CSV，index=False 不存多餘索引

# \bar{X}-R 管制圖統計量（n=4 常數）  # 說明：n=4 時 A2,D3,D4,d2 皆為固定查表常數
subgroup_summary = (  # 開始算每組彙總統計量，括號允許換行
    process.groupby("subgroup")["measurement"]  # 按 subgroup 分組取 measurement 欄
    .agg(xbar="mean", range=lambda x: x.max() - x.min())  # 聚合：xbar取平均，range用 lambda 算全距 max-min
    .reset_index()  # 還原索引，讓 subgroup 變回普通欄
)  # 賦值結束
# 區隔彙總與中心線
xbarbar = subgroup_summary["xbar"].mean()  # Xbar-Bar 總平均，25組平均的再平均，為 Xbar 圖中心線 CL
rbar = subgroup_summary["range"].mean()  # R-Bar 平均全距，25組全距的平均，為 R 圖中心線 CL

A2 = 0.729  # Xbar 界限係數，n=4 查表值，用於 Xbarbar ± A2*Rbar
D3 = 0.000  # R 下界係數，n=4 查表值為0，故 R 下限為0
D4 = 2.282  # R 上界係數，n=4 查表值，用於 D4*Rbar
d2 = 2.059  # 標準差換算係數，n=4 查表值，用於 sigma=Rbar/d2

xbar_ucl = xbarbar + A2 * rbar  # Xbar 上界 UCL = 中心線 + A2×Rbar
xbar_lcl = xbarbar - A2 * rbar  # Xbar 下界 LCL = 中心線 - A2×Rbar
r_ucl = D4 * rbar  # R 上界 UCL = D4×Rbar
r_lcl = D3 * rbar  # R 下界 LCL = D3×Rbar，本例為0

xbar_in_control = (  # 判斷 Xbar 是否全部受控，結果 True/False
    (subgroup_summary["xbar"] <= xbar_ucl)  # 條件1：每組平均 ≤ 上界
    & (subgroup_summary["xbar"] >= xbar_lcl)  # 條件2：且每組平均 ≥ 下界，& 為逐元素且
).all()  # 25組全 True 才回傳 True，一點超界即 False

r_in_control = (  # 判斷 R 是否全部受控，檢查組內變異是否穩定
    (subgroup_summary["range"] <= r_ucl)  # 條件1：每組全距 ≤ R上界
    & (subgroup_summary["range"] >= r_lcl)  # 條件2：且每組全距 ≥ R下界
).all()  # 25組全滿足才 True

stable_by_3sigma = bool(xbar_in_control and r_in_control)  # 總穩定：兩圖皆受控才算穩定，bool()確保純布林值

# 組內標準差估計：R-bar / d2  # 說明：用平均全距反推短期組內 sigma，排除組間漂移，算 Cp/Cpk 用
sigma_within = rbar / d2  # 組內 sigma = Rbar/d2，短期能力，反映製程潛力

# 整體標準差：全部觀測值的樣本標準差  # 說明：不分組對100筆直接算，反映長期整體變異，算 Pp/Ppk 用
overall_mean = process["measurement"].mean()  # 100筆總平均，作為製程中心，後續算 CPU/CPL 用
sigma_overall = process["measurement"].std(ddof=1)  # 100筆樣本標準差，ddof=1 分母用 n-1，為不偏估計

# 常態假設下的能力指標  # 說明：以下公式前提是穩定+近似常態，否則無效
cp = (USL - LSL) / (6 * sigma_within)  # Cp潛在能力：公差寬度/6倍組內 sigma，只看寬窄不看偏移
cpu = (USL - overall_mean) / (3 * sigma_within)  # CPU上單邊能力：平均到USL距離/3倍 sigma，看偏上風險
cpl = (overall_mean - LSL) / (3 * sigma_within)  # CPL下單邊能力：平均到LSL距離/3倍 sigma，看偏下風險
cpk = min(cpu, cpl)  # Cpk取小者，同時考慮寬窄與置中，本例偏上故由 CPU 決定

pp = (USL - LSL) / (6 * sigma_overall)  # Pp長期績效：公式同Cp但用整體 sigma，反映長期實際表現
ppu = (USL - overall_mean) / (3 * sigma_overall)  # PPU上單邊長期績效：平均到USL距離/3倍整體 sigma
ppl = (overall_mean - LSL) / (3 * sigma_overall)  # PPL下單邊長期績效：平均到LSL距離/3倍整體 sigma
ppk = min(ppu, ppl)  # Ppk取PPU/PPL小者，對應 Cpk 的長期版，補上 D段 print 需要的變數

# 以下門檻只作為示範判讀，實務須依客戶／產業規範設定  # 警語：課堂簡化規則，不可直接當放行標準
msa_pass_for_demo = pct_tolerance_grr < 10 and ndc >= 5  # 示範用 MSA 門檻：%Tolerance<10% 且 NDC>=5，引用B段變數，需先跑過B段

if msa_pass_for_demo and stable_by_3sigma:  # 綜合條件：量具示範通過 且 製程三倍標準差穩定，兩者皆 True 才可解讀
    capability_status = "可進行初步能力解讀；仍須檢查常態性、取樣設計及其他管制圖規則"  # 通過時的結論字串，提醒還要驗常態等，補上 D段 print 需要的變數
else:  # 否則：量具太差或製程不穩任一發生
    capability_status = "能力指標僅作示範計算，不宜直接作為製程放行結論"  # 不通過的結論字串，僅供教學計算用
# ===========  # 注意：原文此行缺 # 會報錯，應改為註解，僅為段落分隔線無作用

In [ ]:
# =========================================================  
# D. 輸出結果  # 標題註解：本段將前面 B段MSA 和 C段SPC 算好的變數用 print 印出報表
# =========================================================  
print("\n========== MSA：交叉式 Gage R&R ==========")  # 印出 MSA 報表大標題，\n 表先換行再印
print(f"總量測筆數             : {len(msa)}")  # 印出 msa 表格總列數，f-string 嵌入變數，本例應為60筆
print(f"重複性 SD              : {sd_repeatability:.6f} mm")  # 印出重複性標準差，:.6f 表取小數6位，mm為單位
print(f"作業員差異 SD          : {sd_operator:.6f} mm")  # 印出作業員主效應標準差，反映換人差異
print(f"Part×Operator SD       : {sd_interaction:.6f} mm")  # 印出零件×人員交互作用標準差，反映特定人量特定件的偏差
print(f"總 Gage R&R SD         : {sd_grr:.6f} mm")  # 印出量測系統總標準差 GRR，為後續 %GRR 的分子核心
print(f"零件間 SD              : {sd_part:.6f} mm")  # 印出零件間標準差，反映真實產品差異，越大表示量具解析需求越高
print(f"%Study Variation (GRR) : {pct_study_grr:.2f}%")  # 印出 %StudyVar，:.2f 取小數2位加%號，看GRR佔全變異比例
print(f"%Tolerance (GRR)       : {pct_tolerance_grr:.2f}%")  # 印出 %Tolerance，看 6*SD_GRR 吃掉多少規格公差
print(f"ndc                    : {ndc}")  # 印出 NDC 可區分類別數，不需小數，>=5才表示解析力足夠
print(f"MSA 判讀               : {msa_judgment}")  # 印出 B段依 <10%/<=30%/NDC 規則產生的文字結論

print("\nANOVA 表：")  # 印出小標題，\n 先換行，預告下面要印 ANOVA 數字表
print(anova_table.round(8))  # 印出 anova_table，四捨五入到小數8位，避免平方和科學記號太長難讀

print("\n========== 製程穩定性：X-bar / R ==========")  # 印出 SPC 穩定性報表大標題，\n 先換行
print(f"X-bar 中心線           : {xbarbar:.5f} mm")  # 印出 Xbar 圖中心線 CL=Xbarbar，:.5f 取小數5位
print(f"X-bar LCL / UCL        : {xbar_lcl:.5f} / {xbar_ucl:.5f} mm")  # 同時印出 Xbar 圖下界LCL與上界UCL，用 / 隔開
print(f"R-bar                  : {rbar:.5f} mm")  # 印出 R 圖中心線 CL=Rbar，即平均全距
print(f"R LCL / UCL            : {r_lcl:.5f} / {r_ucl:.5f} mm")  # 同時印出 R 圖下界與上界，本例下界為0
print(f"三倍標準差界限檢查     : {'未發現界外點' if stable_by_3sigma else '有界外點'}")  # 用三元條件式依 stable_by_3sigma 布林值顯示中文結論，True顯示未發現界外點

print("\n========== 製程能力指標 ==========")  # 印出製程能力報表大標題，\n 先換行
print(f"規格 LSL / USL         : {LSL:.3f} / {USL:.3f} mm")  # 印出規格下限與上限，:.3f 取小數3位呼應 19.900/20.100
print(f"目標 Target            : {TARGET:.3f} mm")  # 印出目標值 20.000，作為置中參考
print(f"樣本平均               : {overall_mean:.5f} mm")  # 印出100筆製程數據總平均，看有無偏離目標
print(f"組內標準差估計         : {sigma_within:.6f} mm")  # 印出短期組內標準差 Rbar/d2，算 Cp/Cpk 用此值
print(f"整體標準差             : {sigma_overall:.6f} mm")  # 印出長期整體標準差 std(ddof=1)，算 Pp/Ppk 用此值
print(f"Cp / Cpk               : {cp:.3f} / {cpk:.3f}")  # 同時印出短期潛在能力 Cp 與實際能力 Cpk，:.3f 為能力指標慣用精度
print(f"Pp / Ppk               : {pp:.3f} / {ppk:.3f}")  # 同時印出長期績效 Pp 與 Ppk，注意 ppk 與 capability_status 需在 C段後段先定義否則會報錯
print(f"解讀狀態               : {capability_status}")  # 印出能力綜合文字結論，如是否穩定且 Cpk 達標，需前面先依管制圖+常態檢定賦值

In [ ]:
# =========================================================  
# E. 繪製教學圖：X-bar、R 管制圖及零件×作業員平均圖  # 標題註解：本段一次畫3張圖，驗證製程穩定性與量測交互作用
# =========================================================  
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))  # 建立1列3欄子圖，fig為總圖框、axes為3個座標軸陣列，figsize設寬16高4.5吋

# X-bar chart  # 說明註解：以下 8 行專門畫第1張 X-bar 平均數管制圖
axes[0].plot(  # 開始在第0個子圖畫折線，呼叫 plot
    subgroup_summary["subgroup"],  # 參數1：X軸為子組編號 1~25
    subgroup_summary["xbar"],  # 參數2：Y軸為每組平均值
    marker="o"  # 參數3：每個點用圓圈標記，方便看出界外點
)  # plot() 呼叫結束
axes[0].axhline(xbarbar, color="green", label="CL")  # 畫綠色水平中心線 CL=Xbarbar，label供圖例顯示
axes[0].axhline(xbar_ucl, color="red", linestyle="--", label="UCL/LCL")  # 畫紅色虛線上界 UCL，linestyle=--表虛線，label代表上下界
axes[0].axhline(xbar_lcl, color="red", linestyle="--")  # 畫紅色虛線下界 LCL，不重複加 label 避免圖例重複
axes[0].set_title("X-bar Chart")  # 設定第0圖標題為 X-bar Chart
axes[0].set_xlabel("Subgroup")  # 設定第0圖 X軸標籤為 Subgroup 子組
axes[0].set_ylabel("Subgroup mean (mm)")  # 設定第0圖 Y軸標籤為子組平均，單位 mm
axes[0].legend()  # 顯示第0圖圖例，讓 CL、UCL/LCL 標示可見

# R chart  # 說明註解：以下 8 行專門畫第2張 R 全距管制圖，看組內變異是否穩定
axes[1].plot(  # 開始在第1個子圖畫折線
    subgroup_summary["subgroup"],  # 參數1：X軸同為子組編號 1~25
    subgroup_summary["range"],  # 參數2：Y軸為每組全距 max-min
    marker="o"  # 參數3：圓圈標記各組全距點
)  # plot() 呼叫結束
axes[1].axhline(rbar, color="green", label="CL")  # 畫綠色水平中心線 CL=Rbar 平均全距
axes[1].axhline(r_ucl, color="red", linestyle="--", label="UCL/LCL")  # 畫紅色虛線上界 UCL=D4*Rbar
axes[1].axhline(r_lcl, color="red", linestyle="--")  # 畫紅色虛線下界 LCL=D3*Rbar，本例為0
axes[1].set_title("R Chart")  # 設定第1圖標題為 R Chart
axes[1].set_xlabel("Subgroup")  # 設定第1圖 X軸標籤為 Subgroup
axes[1].set_ylabel("Range (mm)")  # 設定第1圖 Y軸標籤為全距，單位 mm
axes[1].legend()  # 顯示第1圖圖例

# MSA interaction plot：各零件在不同作業員下的平均量測值  # 說明註解：若三條線平行表無交互作用，交叉或分開表有交互作用
interaction_means = (  # 開始計算交互作用平均表，左括號允許換行
    msa.groupby(["part", "operator"])["measurement"]  # 按零件+作業員分組，取出 measurement 欄，共10*3=30組
    .mean()  # 對每組取平均，消除2次重複的隨機誤差，得到每人每件的平均值
    .unstack("operator")  # 將 operator 由縱向索引轉為橫向欄位，變成 P01~P10 為列、A/B/C 為欄的寬表格
)  # interaction_means 賦值括號結束
for operator in operators:  # 迴圈：依序處理 A,B,C 三位作業員，各畫一條線
    axes[2].plot(  # 在第2個子圖畫該作業員的折線
        interaction_means.index,  # 參數1：X軸為零件索引 P01~P10
        interaction_means[operator],  # 參數2：Y軸為該作業員對各零件的平均量測值
        marker="o",  # 參數3：圓圈標記各零件點
        label=f"Operator {operator}"  # 圖例標籤：f-string 動態產生 Operator A/B/C
    )  # plot() 呼叫結束
axes[2].set_title("Part × Operator Means")  # 設定第2圖標題為零件×人員平均圖
axes[2].set_xlabel("Part")  # 設定第2圖 X軸標籤為 Part 零件
axes[2].set_ylabel("Mean measurement (mm)")  # 設定第2圖 Y軸標籤為平均量測值，單位 mm
axes[2].tick_params(axis="x", rotation=45)  # 設定 X軸刻度標籤旋轉45度，避免 P01~P10 文字重疊
axes[2].legend()  # 顯示第2圖圖例，區分 A/B/C 三條線

plt.tight_layout()  # 自動調整3張子圖間距，避免標題、軸標籤互相擠壓重疊
plt.savefig("msa_spc_demo.png", dpi=150)  # 將總圖存檔為 PNG，dpi=150表解析度，越高越清晰檔案越大
plt.show()  # 在螢幕彈出顯示圖形，教學時可直接檢視，若純批次執行可省略

print("\n已輸出：")  # 印出結尾提示標題，\n 先換行，告知使用者產出了哪些檔案
print("  msa_synthetic_data.csv")  # 印出第1個產出檔：A段的60筆 MSA 數據
print("  process_synthetic_data.csv")  # 印出第2個產出檔：C段的100筆製程數據
print("  msa_spc_demo.png")  # 印出第3個產出檔：E段的3合1教學圖